In [ ]:
from plot_utils import *
setup_plot_style()

X_JITTER = 1e-6

ROOT_DIR = TORIC_CODE_ROOT
OUT_DIR = FIGPLOT_DIR
os.makedirs(OUT_DIR, exist_ok=True)

by_L = load_toric_code_data()
print("Found L:", sorted(by_L.keys()))

In [ ]:
# 多 L 的 On-the-fly Fidelity：按 L=3,5,7,9,11 画一行五张子图
PLOT_MWPM = True
PLOT_UF = True
PLOT_OURS = True

QUEKUF_ENABLED = True
QUEKUF_P = 0.001
QUEKUF_CYCLES_MANUAL: Dict[int, Optional[float]] = {3: 207, 5: 988, 7: 2915, 9: 6192, 11: 11760}
QUEKUF_NORM_SCALE: Dict[int, float] = {3: 4.2, 5: 4.2, 7: 4.2, 9: 4.2, 11: 4.2}

fig, axes = plt.subplots(1, len(L_TARGETS), figsize=(22, 4.8), sharey=False)
for i, L in enumerate(L_TARGETS):
    ax = axes[i]
    data = by_L.get(L)
    if data is None:
        ax.set_title(f"Code Distance={L} (N/A)")
        ax.axis('off')
        continue

    ps: List[float] = data.get('ps', [])
    ler: Dict[str, List[float]] = data.get('ler', {})
    latency_means: Dict[str, List[float]] = data.get('latency_means', {})

    ax.set_yscale('log')

    cluster_vals = []

    # Micro-Blossom (MWPM)
    if PLOT_MWPM and isinstance(ler.get('mwpm'), list) and len(ler['mwpm']) == len(ps):
        mwpm_cycles = [get_micro_blossom_cycles(L, p) * SCALE_MICRO_BLOSSOM for p in ps]
        y_mwpm = [1.0 - compute_fidelity(float(l), float(c), d=D_EXPONENT) for l, c in zip(ler['mwpm'], mwpm_cycles)]
        cluster_vals.extend(y_mwpm)
        if PLOT_AS_SCATTER:
            xp = [p - X_JITTER for p in ps]
            ax.scatter(xp, y_mwpm, s=SCATTER_SIZE, marker='o', color='#1f77b4', edgecolor='black', linewidths=MARKER_EDGE_WIDTH, label='Micro-Blossom (MWPM Hardware)')
        else:
            ax.plot(ps, y_mwpm, marker='o', linestyle='-', color='#1f77b4', label='Micro-Blossom (MWPM Hardware)')

    # Helios (UF)
    if PLOT_UF and isinstance(ler.get('uf'), list) and len(ler['uf']) == len(ps):
        uf_cycles = [get_helios_cycles(L, p) * SCALE_HELIOS for p in ps]
        y_uf = [1.0 - compute_fidelity(float(l), float(c), d=D_EXPONENT) for l, c in zip(ler['uf'], uf_cycles)]
        cluster_vals.extend(y_uf)
        if PLOT_AS_SCATTER:
            xp = ps
            ax.scatter(xp, y_uf, s=SCATTER_SIZE, marker='s', color='#ff7f0e', edgecolor='black', linewidths=MARKER_EDGE_WIDTH, label='Helios (UF Hardware)')
        else:
            ax.plot(ps, y_uf, marker='s', linestyle='-', color='#ff7f0e', label='Helios (UF Hardware)')

    # Ours
    if PLOT_OURS and isinstance(ler.get('uf_efficient_votemax'), list) and len(ler['uf_efficient_votemax']) == len(ps):
        eff_total = latency_means.get('efficient_total', [])
        if isinstance(eff_total, list) and len(eff_total) == len(ps) and any(float(v) > 0 for v in eff_total):
            eff_cycles = [float(v) * SCALE_TOTAL_TO_CYCLES for v in eff_total]
            y_ours = [1.0 - compute_fidelity(float(l), float(c), d=D_EXPONENT) for l, c in zip(ler['uf_efficient_votemax'], eff_cycles)]
            cluster_vals.extend(y_ours)
            if PLOT_AS_SCATTER:
                xp = [p + X_JITTER for p in ps]
                ax.scatter(xp, y_ours, s=SCATTER_SIZE, marker='^', color='#2ca02c', edgecolor='black', linewidths=MARKER_EDGE_WIDTH, label='Ours (Hardware)')
            else:
                ax.plot(ps, y_ours, marker='^', linestyle='-', color='#2ca02c', label='Ours (Hardware)')

    _set_cluster_ylim(ax, cluster_vals, pad_decades=0.25)

    # 打印当前 L 的 fidelity 数值
    print(f"\n[Multi-L] L={L} Fidelity values")
    if 'mwpm' in ler:
        mwpm_cycles = [get_micro_blossom_cycles(L, p) * SCALE_MICRO_BLOSSOM for p in ps]
        f_mwpm = [compute_fidelity(float(l), float(c), d=D_EXPONENT) for l, c in zip(ler['mwpm'], mwpm_cycles)]
        print("MWPM:", [f"{v:.6e}" for v in f_mwpm])
    if 'uf' in ler:
        uf_cycles = [get_helios_cycles(L, p) * SCALE_HELIOS for p in ps]
        f_uf = [compute_fidelity(float(l), float(c), d=D_EXPONENT) for l, c in zip(ler['uf'], uf_cycles)]
        print("UF:", [f"{v:.6e}" for v in f_uf])
    if 'uf_efficient_votemax' in ler and isinstance(latency_means.get('efficient_total', []), list) and len(latency_means.get('efficient_total', [])) == len(ps):
        eff_cycles = [float(v) * SCALE_TOTAL_TO_CYCLES for v in latency_means.get('efficient_total', [])]
        f_ours = [compute_fidelity(float(l), float(c), d=D_EXPONENT) for l, c in zip(ler['uf_efficient_votemax'], eff_cycles)]
        print("Ours:", [f"{v:.6e}" for v in f_ours])

    # QUEKUF
    if QUEKUF_ENABLED and isinstance(ler.get('uf'), list):
        try:
            p_idx = ps.index(QUEKUF_P)
        except ValueError:
            p_idx = -1
        if p_idx >= 0 and p_idx < len(ler['uf']):
            cycles_manual = QUEKUF_CYCLES_MANUAL.get(L)
            if cycles_manual is not None:
                ler_val = float(ler['uf'][p_idx])
                cycles = float(cycles_manual) * float(QUEKUF_NORM_SCALE.get(L, 1.0))
                qy = 1.0 - compute_fidelity(ler_val, cycles, d=D_EXPONENT)
                _scatter_with_oob(ax, QUEKUF_P, qy, s=SCATTER_SIZE*1.3, marker='X', color='#9467bd', edgecolor='black', linewidths=MARKER_EDGE_WIDTH, label='QUEKUF (UF hardware)')

    ax.grid(True, which='both', linestyle='--', alpha=0.3)
    desired_ticks = [p for p in ps if p in (0.001, 0.0015)]
    if desired_ticks:
        ax.set_xticks(desired_ticks)
        ax.set_xticklabels([f"{p:.4f}" for p in desired_ticks], rotation=0, ha='center')
    ax.text(0.95, 0.02, f"Code Distance {L}", transform=ax.transAxes,
            ha='right', va='bottom', fontsize=18)

axes[0].set_ylabel('System Infidelity', fontsize=24)
for ax in axes:
    ax.tick_params(axis='x', labelrotation=0)

axes[-1].annotate('Better', xy=(0.92, 0.15), xytext=(0.92, 0.55),
                  xycoords='axes fraction', textcoords='axes fraction',
                  ha='center', va='top', fontsize=16,
                  arrowprops=dict(arrowstyle='-|>', lw=3.0, color='black'))

# 汇总图例
handles_dict = {}
for ax in axes:
    h, l = ax.get_legend_handles_labels()
    for handle, label in zip(h, l):
        if label and label not in handles_dict:
            handles_dict[label] = handle
if handles_dict:
    desired_order = ['Micro-Blossom (MWPM Hardware)', 'Helios (UF Hardware)', 'Ours (Hardware)']
    others = [lbl for lbl in handles_dict.keys() if lbl not in desired_order]
    ordered_labels = [lbl for lbl in desired_order if lbl != 'Ours (Hardware)' and lbl in handles_dict] \
                     + others \
                     + (['Ours (Hardware)'] if 'Ours (Hardware)' in handles_dict else [])
    ordered_handles = [handles_dict[lbl] for lbl in ordered_labels]

    leg = fig.legend(
        ordered_handles, ordered_labels,
        loc='upper center', ncol=len(ordered_labels), frameon=True,
        bbox_to_anchor=(0.5, 1.05)
    )
    frame = leg.get_frame()
    frame.set_edgecolor('black')
    frame.set_linewidth(1.8)
    frame.set_facecolor('white')
    frame.set_alpha(1.0)

fig.supxlabel('Physical Error Rate', fontsize=24, y=0.06)
fig.tight_layout(rect=[0, 0.002, 1, 0.98])

save_path = os.path.join(OUT_DIR, 'on_the_fly_fidelity_row.pdf')
fig.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()
print(f"Saved: {save_path}")